# Age Classification - Fine-tuning EfficientNet-B0

**Input:** Manually labeled face crops (Adult/Child)
**Output:** Trained classifier + Confusion Matrix

## Step 1: Install Dependencies

In [ ]:
!pip install torch torchvision timm scikit-learn seaborn -q
print("✓ Dependencies installed")

✓ Dependencies installed


## Step 2: Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted")

Mounted at /content/drive
✓ Drive mounted


## Step 3: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using: {device}")

✓ Using: cpu


## Step 4: Configuration

In [ ]:
BASE_PATH = Path("/content/drive/MyDrive/face_pipeline_project")
LABELS_FILE = BASE_PATH / "ground_truth/age_labels_manual.csv"
MODEL_SAVE_PATH = BASE_PATH / "models/age_classifier.pth"
FACE_CROPS_FOLDER = BASE_PATH / "face_crops_aligned"
CLASSIFIED_FOLDER = BASE_PATH / "classified"
METRICS_FOLDER = BASE_PATH / "metrics"

# Training config
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224

# Create folders
MODEL_SAVE_PATH.parent.mkdir(exist_ok=True, parents=True)
(CLASSIFIED_FOLDER / "adult").mkdir(exist_ok=True, parents=True)
(CLASSIFIED_FOLDER / "child").mkdir(exist_ok=True, parents=True)

print(f"Labels: {LABELS_FILE}")
print(f"Model will save to: {MODEL_SAVE_PATH}")

Labels: /content/drive/MyDrive/face_pipeline_project/ground_truth/age_labels_manual.csv
Model will save to: /content/drive/MyDrive/face_pipeline_project/models/age_classifier.pth


## Step 5: Load & Clean Labels

In [ ]:
df = pd.read_csv(LABELS_FILE)
print(f"Loaded: {len(df)} rows")

# Remove duplicates
df = df.drop_duplicates(subset=['image_path'], keep='last')
print(f"After dedup: {len(df)} rows")

# Filter to Adult and Child only (exclude not_face)
df = df[df['label'].isin(['adult', 'child'])].reset_index(drop=True)
print(f"Adult + Child only: {len(df)} rows")

# Convert to binary labels
df['label_int'] = (df['label'] == 'adult').astype(int)  # 1=adult, 0=child

print(f"\nClass distribution:")
print(df['label'].value_counts())

Loaded: 1290 rows
After dedup: 1290 rows
Adult + Child only: 1283 rows

Class distribution:
label
adult    1033
child     250
Name: count, dtype: int64


## Step 6: Train/Val Split

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_int']
)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")
print(f"\nTrain distribution:")
print(train_df['label'].value_counts())
print(f"\nVal distribution:")
print(val_df['label'].value_counts())

Train: 1026 | Val: 257

Train distribution:
label
adult    826
child    200
Name: count, dtype: int64

Val distribution:
label
adult    207
child     50
Name: count, dtype: int64


## Step 7: Dataset & DataLoader

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        label = row['label_int']

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = FaceDataset(train_df, train_transform)
val_dataset = FaceDataset(val_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✓ DataLoaders ready")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

✓ DataLoaders ready
  Train batches: 33
  Val batches: 9


## Step 8: Build Model

In [ ]:
# Load pretrained EfficientNet-B0
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)
model = model.to(device)

print(f"✓ EfficientNet-B0 loaded")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

✓ EfficientNet-B0 loaded
  Parameters: 4,010,110


## Step 9: Training Setup

In [ ]:
# Class weights for imbalance
class_counts = train_df['label_int'].value_counts().sort_index()
weights = 1.0 / class_counts.values
weights = weights / weights.sum()
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
print(f"Class weights: {class_weights}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("✓ Training setup complete")

Class weights: tensor([0.8051, 0.1949])
✓ Training setup complete


## Step 10: Training Loop

In [ ]:
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = val_correct / val_total
    scheduler.step()

    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  ✓ Saved best model (val_acc: {val_acc:.2%})")

    history['train_loss'].append(train_loss / len(train_loader))
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss / len(val_loader))
    history['val_acc'].append(val_acc)

    print(f"  Train: {train_acc:.2%} | Val: {val_acc:.2%}")

print(f"\n✅ Training complete! Best val acc: {best_val_acc:.2%}")

Epoch 1/15: 100%|██████████| 33/33 [06:10<00:00, 11.24s/it]


  ✓ Saved best model (val_acc: 85.60%)
  Train: 73.00% | Val: 85.60%


Epoch 2/15:  18%|█▊        | 6/33 [01:01<04:35, 10.20s/it]


KeyboardInterrupt: 

## Step 11: Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.set_title('Loss')

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.set_title('Accuracy')

plt.tight_layout()
plt.savefig(METRICS_FOLDER / "training_curves.png", dpi=150)
plt.show()

## Step 12: Confusion Matrix on Validation

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Child', 'Adult'],
            yticklabels=['Child', 'Adult'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Age Classification Confusion Matrix (Fine-tuned)')
plt.savefig(METRICS_FOLDER / "confusion_matrix_age_finetuned.png", dpi=150)
plt.show()

print(classification_report(all_labels, all_preds, target_names=['Child', 'Adult']))

## Step 13: Run Inference on ALL Face Crops

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

# Collect all face crops
all_crops = []
for video_folder in FACE_CROPS_FOLDER.iterdir():
    if video_folder.is_dir():
        for face_file in video_folder.glob("*.jpg"):
            all_crops.append({
                'video': video_folder.name,
                'face_file': face_file.name,
                'image_path': str(face_file)
            })

print(f"Total face crops: {len(all_crops)}")

# Inference
results = []
adult_count = 0
child_count = 0

for crop in tqdm(all_crops, desc="Classifying"):
    try:
        img = Image.open(crop['image_path']).convert('RGB')
        img_tensor = val_transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(img_tensor)
            prob = torch.softmax(output, dim=1)
            predicted = output.argmax(dim=1).item()
            confidence = prob[0, predicted].item()

        classification = 'adult' if predicted == 1 else 'child'

        # Copy to classified folder
        dest_folder = CLASSIFIED_FOLDER / classification / crop['video']
        dest_folder.mkdir(exist_ok=True, parents=True)
        shutil.copy(crop['image_path'], dest_folder / crop['face_file'])

        if classification == 'adult':
            adult_count += 1
        else:
            child_count += 1

        results.append({
            'video': crop['video'],
            'face_file': crop['face_file'],
            'classification': classification,
            'confidence': round(confidence, 3)
        })

    except Exception as e:
        print(f"Error: {crop['face_file']} - {e}")

# Save results
pd.DataFrame(results).to_csv(METRICS_FOLDER / "age_classifications_finetuned.csv", index=False)

print(f"\n✅ Complete!")
print(f"Adults: {adult_count}")
print(f"Children: {child_count}")

---
## ✅ Phase 4 Complete!

**Outputs:**
- Model: `/models/age_classifier.pth`
- Adult faces: `/classified/adult/`
- Child faces: `/classified/child/`
- Results: `/metrics/age_classifications_finetuned.csv`

**Next: Phase 5 - Emotion Classification (Child faces only)**